# 05 — Genetic Algorithm Feature Selection

From-scratch **binary GA**. Each chromosome is a feature mask. Fitness = stratified CV **F1** of the small Random Forest − λ × (selected / total).

### Evolutionary narrative
- **Selection** pressures the population toward high-F1 subsets (exploitation).
- **Crossover** recombines building blocks of good masks (schema theorem intuition).
- **Mutation** maintains diversity so the search does not collapse early (exploration).
- **Elitism** preserves the best chromosome each generation.

In [1]:
from pathlib import Path
import random

import numpy as np

RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

cwd = Path.cwd().resolve()
PROJECT_ROOT = next(
    (p for p in [cwd, *cwd.parents] if (p / "environment.yml").exists()),
    cwd,
)

DATA_RAW = PROJECT_ROOT / "data" / "raw"
DATA_INTERIM = PROJECT_ROOT / "data" / "interim"
DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
FIGURES_DIR = PROJECT_ROOT / "reports" / "figures"
REPORTS_DIR = PROJECT_ROOT / "reports"
MODELS_DIR = PROJECT_ROOT / "models"

for d in (DATA_INTERIM, DATA_PROCESSED, FIGURES_DIR, MODELS_DIR):
    d.mkdir(parents=True, exist_ok=True)

print(f"Project root : {PROJECT_ROOT}")
print(f"Random seed  : {RANDOM_SEED}")

Project root : C:\Users\ishan\OneDrive\Documents\Work\3 Year 2 Semester\IT41033 - Nature Inspired Algorithms\Mini Project\ecommerce-customer-churn-prediction
Random seed  : 42


In [2]:
import json
import time
import joblib
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import StratifiedKFold, cross_val_score

X_train = pd.read_csv(DATA_PROCESSED / "X_train.csv")
y_train = pd.read_csv(DATA_PROCESSED / "y_train.csv").squeeze().values
feature_names = np.array(X_train.columns)
X = X_train.values
n_features = X.shape[1]
print("n_features", n_features)

n_features 33


In [3]:
# GA hyperparameters
POP_SIZE = 20
N_GENERATIONS = 25
CROSSOVER_RATE = 0.8
MUTATION_RATE = 0.05
TOURNAMENT_K = 3
ELITE = 2
LAMBDA_SPARSITY = 0.05  # complexity penalty weight
CV_SPLITS = 3

rng = np.random.default_rng(RANDOM_SEED)

def make_rf():
    return RandomForestClassifier(
        n_estimators=60, max_depth=6, min_samples_leaf=5,
        class_weight="balanced", random_state=RANDOM_SEED, n_jobs=-1,
    )

def fitness(mask: np.ndarray) -> float:
    mask = mask.astype(bool)
    if mask.sum() == 0:
        return 0.0
    X_sub = X[:, mask]
    cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
    scores = cross_val_score(make_rf(), X_sub, y_train, cv=cv, scoring="f1")
    f1 = float(scores.mean())
    penalty = LAMBDA_SPARSITY * (mask.sum() / n_features)
    return f1 - penalty

def init_population():
    # Bias toward ~50% features on, ensure at least 1
    pop = rng.integers(0, 2, size=(POP_SIZE, n_features))
    for i in range(POP_SIZE):
        if pop[i].sum() == 0:
            pop[i, rng.integers(0, n_features)] = 1
    return pop

def tournament_select(pop, fits):
    idx = rng.choice(len(pop), size=TOURNAMENT_K, replace=False)
    best = idx[np.argmax(fits[idx])]
    return pop[best].copy()

def crossover(p1, p2):
    if rng.random() > CROSSOVER_RATE:
        return p1.copy(), p2.copy()
    point = rng.integers(1, n_features)
    c1 = np.concatenate([p1[:point], p2[point:]])
    c2 = np.concatenate([p2[:point], p1[point:]])
    return c1, c2

def mutate(chrom):
    flips = rng.random(n_features) < MUTATION_RATE
    chrom = chrom.copy()
    chrom[flips] = 1 - chrom[flips]
    if chrom.sum() == 0:
        chrom[rng.integers(0, n_features)] = 1
    return chrom

In [4]:
t0 = time.perf_counter()
population = init_population()
best_fitness_hist = []
mean_fitness_hist = []
best_chrom = None
best_fit = -np.inf

for gen in range(N_GENERATIONS):
    fits = np.array([fitness(ind) for ind in population])
    gen_best_idx = int(np.argmax(fits))
    if fits[gen_best_idx] > best_fit:
        best_fit = float(fits[gen_best_idx])
        best_chrom = population[gen_best_idx].copy()
    best_fitness_hist.append(float(fits.max()))
    mean_fitness_hist.append(float(fits.mean()))
    print(f"Gen {gen:02d} | best={fits.max():.4f} mean={fits.mean():.4f} bits={population[gen_best_idx].sum()}")

    # Next generation
    elite_idx = np.argsort(fits)[-ELITE:]
    new_pop = [population[i].copy() for i in elite_idx]
    while len(new_pop) < POP_SIZE:
        p1 = tournament_select(population, fits)
        p2 = tournament_select(population, fits)
        c1, c2 = crossover(p1, p2)
        new_pop.append(mutate(c1))
        if len(new_pop) < POP_SIZE:
            new_pop.append(mutate(c2))
    population = np.array(new_pop[:POP_SIZE])

ga_time = time.perf_counter() - t0
print(f"GA finished in {ga_time:.1f}s | best fitness={best_fit:.4f} | n_selected={int(best_chrom.sum())}")

Gen 00 | best=0.6132 mean=0.5596 bits=18


Gen 01 | best=0.6200 mean=0.5932 bits=17


Gen 02 | best=0.6418 mean=0.6128 bits=18


Gen 03 | best=0.6418 mean=0.6215 bits=18


Gen 04 | best=0.6441 mean=0.6228 bits=16


...

Gen 20 | best=0.6629 mean=0.6461 bits=17


Gen 21 | best=0.6629 mean=0.6467 bits=17


Gen 22 | best=0.6629 mean=0.6514 bits=17


Gen 23 | best=0.6629 mean=0.6427 bits=17


Gen 24 | best=0.6629 mean=0.6417 bits=17
GA finished in 236.8s | best fitness=0.6629 | n_selected=17


In [5]:
selected_features = feature_names[best_chrom.astype(bool)].tolist()
print("Selected", len(selected_features), "features:")
print(selected_features)

# Full vs GA subset CV F1 (no sparsity penalty — pure performance)
cv = StratifiedKFold(n_splits=CV_SPLITS, shuffle=True, random_state=RANDOM_SEED)
f1_full = float(cross_val_score(make_rf(), X, y_train, cv=cv, scoring="f1").mean())
f1_ga = float(cross_val_score(make_rf(), X[:, best_chrom.astype(bool)], y_train, cv=cv, scoring="f1").mean())

compare = {
    "algorithm": "GA",
    "n_total_features": int(n_features),
    "n_selected": int(best_chrom.sum()),
    "reduction_pct": float(100 * (1 - best_chrom.sum() / n_features)),
    "best_fitness": best_fit,
    "cv_f1_full": f1_full,
    "cv_f1_selected": f1_ga,
    "runtime_sec": ga_time,
    "selected_features": selected_features,
    "mask": best_chrom.astype(int).tolist(),
    "hyperparams": {
        "POP_SIZE": POP_SIZE, "N_GENERATIONS": N_GENERATIONS,
        "CROSSOVER_RATE": CROSSOVER_RATE, "MUTATION_RATE": MUTATION_RATE,
        "LAMBDA_SPARSITY": LAMBDA_SPARSITY, "CV_SPLITS": CV_SPLITS,
    },
}
(DATA_PROCESSED / "ga_feature_selection.json").write_text(json.dumps(compare, indent=2), encoding="utf-8")
pd.Series(selected_features).to_csv(DATA_PROCESSED / "ga_selected_features.csv", index=False, header=["feature"])
print(json.dumps({k: v for k, v in compare.items() if k not in ("selected_features", "mask")}, indent=2))

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(best_fitness_hist, label="best")
ax.plot(mean_fitness_hist, label="mean")
ax.set_xlabel("Generation")
ax.set_ylabel("Fitness (CV F1 - penalty)")
ax.set_title("GA fitness history")
ax.legend()
fig.tight_layout()
fig.savefig(FIGURES_DIR / "05_ga_fitness_history.png", dpi=150)
plt.show()

Selected 17 features:
['Tenure', 'WarehouseToHome', 'NumberOfAddress', 'OrderAmountHikeFromlastYear', 'DaySinceLastOrder', 'CityTier', 'SatisfactionScore', 'Complain', 'PreferredLoginDevice_Computer', 'PreferredPaymentMode_Credit Card', 'PreferredPaymentMode_Debit Card', 'Gender_Female', 'PreferedOrderCat_Fashion', 'PreferedOrderCat_Laptop & Accessory', 'PreferedOrderCat_Mobile Phone', 'PreferedOrderCat_Others', 'MaritalStatus_Single']


{
  "algorithm": "GA",
  "n_total_features": 33,
  "n_selected": 17,
  "reduction_pct": 48.484848484848484,
  "best_fitness": 0.6628923024249096,
  "cv_f1_full": 0.6452820173330552,
  "cv_f1_selected": 0.6886498781824854,
  "runtime_sec": 236.8041946000012,
  "hyperparams": {
    "POP_SIZE": 20,
    "N_GENERATIONS": 25,
    "CROSSOVER_RATE": 0.8,
    "MUTATION_RATE": 0.05,
    "LAMBDA_SPARSITY": 0.05,
    "CV_SPLITS": 3
  }
}


C:\Users\ishan\AppData\Local\Temp\ipykernel_14980\188220289.py:40: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


**Insight:** If CV F1 on the GA subset is close to (or better than) the full set while using fewer features, GA succeeded at **feature reduction** without sacrificing predictive signal — exactly what we want before training TabNet.

**Next:** Notebook `06` — binary PSO with the same fitness for a fair comparison.